In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import MNIST

In [2]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307), (0.3081))    
])

train_set = MNIST(root="./data", train=True, download=True, transform=transform)
test_set = MNIST(root="./data", train=False, download=True, transform=transform)

In [3]:
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)

In [4]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        # Sequential = run these layers one after another automatically
        self.conv_layers = nn.Sequential(
            
            # Conv2d: slide 32 filters (3x3) over 1-channel image to detect features
            # padding=1 keeps image size same (28x28 stays 28x28)
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            # ReLU: kill negative values, keep positive detections only
            nn.ReLU(),
            # MaxPool: take best value from each 2x2 block, halves size 28x28 -> 14x14
            nn.MaxPool2d(2, 2),

            # Conv2d: 64 filters now detecting more complex features from 32 feature maps
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            # 14x14 -> 7x7
            nn.MaxPool2d(2, 2),

            # Conv2d: 128 filters detecting even deeper/complex patterns
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            # 7x7 -> 3x3 (floors 3.5)
            nn.MaxPool2d(2, 2)
        )

        # fc_layers: fully connected layers for final classification
        # takes flattened 3x3x128=1152 vector and maps to 10 digit classes
        self.fc_layers = nn.Sequential(
            # Linear: learns which feature combinations matter
            nn.Linear(3*3*128, 256),
            nn.ReLU(),
            # Final layer: 10 outputs = 10 digit classes (no ReLU, keep raw logits)
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # flattening -> 64, 28,3,3 -> 64, 1152
        x = self.fc_layers(x) # keep repeating

        return x

In [5]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters()) #parameters(weight, bias)

In [6]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()
        output = model.forward(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()
        
    print(f"Epoch [{epoch+1}/{epochs}] | Loss: {epoch_training_loss/len(train_loader):.4f}")

Epoch [1/10] | Loss: 0.1445
Epoch [2/10] | Loss: 0.0404
Epoch [3/10] | Loss: 0.0292
Epoch [4/10] | Loss: 0.0230
Epoch [5/10] | Loss: 0.0175
Epoch [6/10] | Loss: 0.0149
Epoch [7/10] | Loss: 0.0129
Epoch [8/10] | Loss: 0.0111
Epoch [9/10] | Loss: 0.0113
Epoch [10/10] | Loss: 0.0084


In [7]:
correct_labels = 0
total_labels = 0
model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"Accuracy = {correct_labels/total_labels*100}")

Accuracy = 99.1


In [ ]:
import torch.nn as nn
import torch.optim as optim

In [8]:
# RNN
class RNN(nn.Module):
    def __init__(self, input_size=28, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        self.fc = nn.Linear(hidden_size, 10)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        out, _ = self.rnn(x, h0)

        out = self.fc(out[:, -1])
        return out


In [9]:
model = RNN(input_size=28)
criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [16]:
epochs = 10

for epoch in range(epochs):
    model.train()
    epoch_training_los=0.0
    
    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.squeeze(1)
        outputs = model(Xb)

        loss = criteria(outputs, yb)
        loss.backward()
        optimizer.step()
        epoch_training_los += loss.item()
    
    print(f"Epoch [{epoch+1}/{epochs}] | Loss: {epoch_training_los/len(train_loader):.4f}")

Epoch [1/10] | Loss: 0.1295
Epoch [2/10] | Loss: 0.1233
Epoch [3/10] | Loss: 0.1166
Epoch [4/10] | Loss: 0.1057
Epoch [5/10] | Loss: 0.1092
Epoch [6/10] | Loss: 0.1061
Epoch [7/10] | Loss: 0.0975
Epoch [8/10] | Loss: 0.1010
Epoch [9/10] | Loss: 0.1009
Epoch [10/10] | Loss: 0.0956


In [ ]:
model.eval()

with torch.no_grad():
    correct_vals = 0
    total_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.squeeze(1)

        outputs = model(Xb)
        _, predicted = torch.max(outputs, 1)

        total_vals += yb.size(0)
        correct_vals +=(predicted == yb).sum().item()

print(f"Accuracy: {correct_vals/total_vals*100:.2f}%")


Accuracy: 95.95%
